# 313 — Compliance Agent

Demonstrates the `ComplianceAgent` assessing real loan applications against APRA standards.

**Real findings expected:**
- `LOAN-0002`: LVR=95% → breaches APG-223-THR-008 (≥90% requires senior review)
- `LOAN-0013`: LVR=92% → same threshold
- `LOAN-0007`: LVR=80% → borderline, serviceability buffer check required

In [1]:
%run 311_agent_setup.ipynb

/opt/anaconda3/envs/graphrag-finserv/lib/python3.11/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Environment loaded.


2026-03-09 14:46:11,043 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io


Connected to Neo4j.

Node counts in graph:
  {'labels': {'CommercialSecured': 2, 'Federal': 1, 'Address': 609, 'Residential': 582, 'Corporate': 31, 'Assessment': 2, 'Collateral': 466, 'ResidentialSecured': 464, 'Industry': 14, 'National': 5, 'PersonalTransaction': 610, 'Threshold': 157, 'CorporateCurrent': 6, 'Borrower': 628, 'BusinessTransaction': 175, 'SpecialAdminRegion': 1, 'BankAccount': 791, 'Chunk': 205, 'Section': 118, 'Jurisdiction': 7, 'Commercial': 27, 'Requirement': 194, 'Transaction': 173, 'LoanApplication': 466, 'Individual': 597, 'Finding': 9, 'Regulation': 3, 'ReasoningStep': 13, 'Officer': 19}}
  ✓  Borrower: 628
  ✓  LoanApplication: 466
  ✓  BankAccount: 791
  ✓  Transaction: 173
  ✓  Regulation: 3
  ✓  Section: 118
  ✓  Requirement: 194
  ✓  Threshold: 157
  ✓  Chunk: 205
Anthropic client ready. Model: claude-sonnet-4-6
OpenAI client ready.
Tool registry: 2 Neo4j MCP + 5 FastMCP = 7 total
  read-neo4j-cypher
  write-neo4j-cypher
  traverse_compliance_path
  retrieve

In [2]:
from src.agent.compliance_agent import ComplianceAgent

agent = ComplianceAgent(tools=TOOLS, execute_tool_fn=execute_tool)

## Example 1: High LVR loan — LOAN-0002 (LVR=95)

In [3]:
result = agent.run('Is LOAN-0002 compliant with APG-223?')
print(f'Verdict:    {result.verdict}')
print(f'Confidence: {result.confidence}')
print(f'Requirements: {result.requirement_ids}')
print(f'Thresholds breached: {result.threshold_breaches}')
print(f'Assessment ID: {result.assessment_id}')
print(f'\nReasoning steps:')
for i, step in enumerate(result.reasoning_steps, 1):
    print(f'  {i}. {step}')
print(f'\nCypher queries used: {len(result.cypher_used)}')
for q in result.cypher_used:
    print(f'  --- {q[:120]}...')

2026-03-09 14:46:48,246 [INFO] src.agent.compliance_agent: ComplianceAgent: Is LOAN-0002 compliant with APG-223?
2026-03-09 14:46:53,317 [INFO] src.agent.compliance_agent: Tool: traverse_compliance_path(['entity_id', 'entity_type', 'regulation_id'])
2026-03-09 14:46:53,341 [INFO] execute_tool: Tool: traverse_compliance_path | inputs: ['entity_id', 'entity_type', 'regulation_id']
2026-03-09 14:46:53,789 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io
2026-03-09 14:46:53,943 [INFO] src.graph.connection: Neo4j connection closed.
2026-03-09 14:46:53,944 [INFO] src.agent.compliance_agent: Tool: read-neo4j-cypher(['query'])
2026-03-09 14:46:53,944 [INFO] execute_tool: Tool: read-neo4j-cypher | inputs: ['query']
2026-03-09 14:47:01,018 [INFO] src.agent.compliance_agent: Tool: detect_graph_anomalies(['pattern_name'])
2026-03-09 14:47:01,030 [INFO] execute_tool: Tool: detect_graph_anomalies | inputs: ['pattern_name']
2026-03-09 14:47:01,239 [INFO] src.gr

Verdict:    NON_COMPLIANT
Confidence: 0.93
Requirements: ['APG-223-REQ-040', 'APG-223-REQ-042', 'APG-223-REQ-015', 'APG-223-REQ-016', 'APG-223-REQ-020', 'APG-223-REQ-021', 'APG-223-REQ-023', 'APG-223-REQ-044', 'APG-223-REQ-053', 'APG-223-REQ-054']
Thresholds breached: [{'threshold_id': 'APG-223-THR-008 (LVR 95% ≥ 90%)'}]
Assessment ID: ASSESS-LOAN-0002-APG-223-2026-03-09

Reasoning steps:
  1. **Escalate immediately** to senior management for formal high-LVR review of LOAN-0002 per APG-223-REQ-040, with documented Board oversight.
  2. **Obtain or arrange LMI coverage** before approval, or document alternative risk mitigants (e.g., risk-adjusted pricing, higher provisioning) per APG-223-REQ-042.
  3. **Verify serviceability buffer** — confirm the ADI's calculator applied a minimum 8.11% assessment rate (5.11% + 3.0%) across all debt commitments.
  4. **Obtain and verify income documentation** for Katarina Wu; apply ≥20% haircut to any non-salary income per APG-223-THR-006.
  5. **Portf

## Example 2: Near-threshold loan — LOAN-0007 (LVR=80)

In [4]:
result2 = agent.run(
    'Assess LOAN-0007 against APG-223 serviceability requirements. '
    'The borrower has no declared non-salary income.'
)
print(f'Verdict:    {result2.verdict}')
print(f'Confidence: {result2.confidence}')
print(f'Thresholds breached: {result2.threshold_breaches}')

2026-03-09 14:47:59,121 [INFO] src.agent.compliance_agent: ComplianceAgent: Assess LOAN-0007 against APG-223 serviceability requirements. The borrower has no declared non-salary income.
2026-03-09 14:48:05,838 [INFO] src.agent.compliance_agent: Tool: traverse_compliance_path(['entity_id', 'entity_type', 'regulation_id'])
2026-03-09 14:48:05,920 [INFO] execute_tool: Tool: traverse_compliance_path | inputs: ['entity_id', 'entity_type', 'regulation_id']
2026-03-09 14:48:07,185 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io
2026-03-09 14:48:07,322 [INFO] src.graph.connection: Neo4j connection closed.
2026-03-09 14:48:07,330 [INFO] src.agent.compliance_agent: Tool: read-neo4j-cypher(['query'])
2026-03-09 14:48:07,335 [INFO] execute_tool: Tool: read-neo4j-cypher | inputs: ['query']
2026-03-09 14:48:07,536 [INFO] src.agent.compliance_agent: Tool: retrieve_regulatory_chunks(['query_text', 'regulation_id', 'top_k'])
2026-03-09 14:48:07,536 [INFO] execut

Verdict:    NON_COMPLIANT
Confidence: 0.91
Thresholds breached: [{'threshold_id': 'APG-223-THR-003 (serviceability buffer unassessable — zero income base)'}, {'threshold_id': 'APG-223-THR-006 (non-salary income haircut — not applicable but income gap is critical)'}]


## Example 3: Regulation scoping — what applies to residential secured loans?

In [5]:
result3 = agent.run(
    'Which APRA quantitative thresholds apply to residential secured loans '
    'in the Australian federal jurisdiction? List them with values.'
)
print(f'Verdict: {result3.verdict}')
print(f'Requirements: {result3.requirement_ids}')

2026-03-09 14:49:29,867 [INFO] src.agent.compliance_agent: ComplianceAgent: Which APRA quantitative thresholds apply to residential secured loans in the Australian federal jurisdiction? List them with values.
2026-03-09 14:49:35,212 [INFO] src.agent.compliance_agent: Tool: read-neo4j-cypher(['query'])
2026-03-09 14:49:35,214 [INFO] execute_tool: Tool: read-neo4j-cypher | inputs: ['query']
2026-03-09 14:49:41,102 [INFO] src.agent.compliance_agent: Tool: read-neo4j-cypher(['query'])
2026-03-09 14:49:41,106 [INFO] execute_tool: Tool: read-neo4j-cypher | inputs: ['query']
2026-03-09 14:49:41,620 [INFO] anthropic._base_client: Retrying request to /v1/messages in 32.000000 seconds


Verdict: INFORMATIONAL
Requirements: ['APG-223-REQ-005', 'APG-223-REQ-012', 'APG-223-REQ-015', 'APG-223-REQ-017', 'APG-223-REQ-019', 'APG-223-REQ-020', 'APG-223-REQ-021', 'APG-223-REQ-040', 'APS-112-REQ-022', 'APS-112-REQ-027', 'APS-112-REQ-028', 'APS-112-REQ-029', 'APS-112-REQ-030', 'APS-112-REQ-031', 'APS-112-REQ-032', 'APS-112-REQ-036', 'APS-220-REQ-041', 'APS-220-REQ-042']


## Layer 3 verification

Confirm Assessment nodes were written to the graph.

In [6]:
assessments = execute_tool('read-neo4j-cypher', {
    'query': """
        MATCH (a:Assessment)
        OPTIONAL MATCH (a)-[:HAS_FINDING]->(f:Finding)
        RETURN a.assessment_id AS id, a.verdict AS verdict,
               a.confidence AS confidence, count(f) AS finding_count
        LIMIT 10
    """
})
import json
print(json.dumps(assessments, indent=2, default=str))

2026-03-09 14:50:57,367 [INFO] execute_tool: Tool: read-neo4j-cypher | inputs: ['query']


{
  "rows": [
    {
      "id": "ASSESS-LOAN-0002-APG-223-2026-03-09",
      "verdict": "NON_COMPLIANT",
      "confidence": 0.93,
      "finding_count": 4
    },
    {
      "id": "ASSESS-LOAN-0007-APG-223-2026-03-09",
      "verdict": "NON_COMPLIANT",
      "confidence": 0.91,
      "finding_count": 5
    }
  ]
}
